# Contract Keyword Extraction Report

Notebook này đọc các file JSON trong `data/output` và tạo bảng so sánh kết quả chạy pipeline: thời gian xử lý, số trang, số segment, số keyword group, số batch LLM và nội dung extracted information.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime

import pandas as pd

try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    def display(value):
        if hasattr(value, "to_string"):
            print(value.to_string(index=False))
        else:
            print(value)

    def Markdown(value):
        return value

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_rows", 200)

## Load Result Files

In [2]:
def clean(value):
    return " ".join(str(value or "").split())


def short(value, limit=260):
    text = clean(value)
    if len(text) <= limit:
        return text
    return text[: limit - 3].rstrip() + "..."


def grouped_keywords(group):
    representative = clean(group.get("representative_keyword"))
    values = [representative, *group.get("related_keywords", [])]
    keywords = []
    seen = set()
    for value in values:
        text = clean(value)
        key = text.casefold()
        if text and key not in seen:
            seen.add(key)
            keywords.append(text)
    return ", ".join(keywords)


def page_text(group):
    metadata = group.get("metadata") if isinstance(group.get("metadata"), dict) else {}
    return clean(metadata.get("page"))


def load_results(output_dir=OUTPUT_DIR):
    rows = []
    for path in sorted(output_dir.glob("*.json")):
        data = json.loads(path.read_text(encoding="utf-8"))
        rows.append(
            {
                "path": path,
                "file_name": path.name,
                "modified_at": datetime.fromtimestamp(path.stat().st_mtime),
                "data": data,
            }
        )
    return rows


results = load_results()
display(Markdown(f"Loaded {len(results)} result file(s) from `{OUTPUT_DIR}`."))

Loaded 7 result file(s) from `D:\deadlinefsoft\extract_information_pipeline\data\output`.

## Run Summary Table

In [3]:
summary_rows = []
for item in results:
    data = item["data"]
    calls = data.get("llm_calls", {})
    summary_rows.append(
        {
            "Result file": item["file_name"],
            "Document": data.get("document_name"),
            "Processing time (s)": data.get("processing_time_seconds"),
            "Pages": data.get("total_pages"),
            "Segments": data.get("total_segments"),
            "Keyword groups": data.get("total_keyword_groups", len(data.get("keyword_groups", []))),
            "LLM1 batches": calls.get("keyword_extraction_batches"),
            "LLM2 batches": calls.get("evidence_extraction_batches"),
            "Last modified": item["modified_at"].strftime("%Y-%m-%d %H:%M:%S"),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Result file,Document,Processing time (s),Pages,Segments,Keyword groups,LLM1 batches,LLM2 batches,Last modified
0,1_original_long_result.json,1_original_long.pdf,96.36,5,64,19,2,1,2026-07-05 23:20:45
1,1_result.json,1.pdf,110.66,2,14,18,1,2,2026-07-05 23:11:42
2,2_result.json,2.pdf,49.62,2,15,11,1,1,2026-07-05 23:57:28
3,4_result.json,4.pdf,44.63,2,12,12,1,1,2026-07-06 00:00:28
4,5_result.json,5.pdf,413.76,16,91,64,2,8,2026-07-05 22:26:46
5,6_result.json,6.pdf,168.69,5,45,15,1,2,2026-07-05 21:00:36
6,agape_atp_supply_result.json,agape_atp_supply.txt,50.30,1,5,8,1,1,2026-07-06 00:06:23


### Evaluation
Hiện tại pipeline đã xử lý được nhiều loại document và sinh ra result file đầy đủ. Tuy nhiên, kết quả cho thấy hệ thống với thời gian xử lý chưa ổn định, file 5.pdf mất hơn 400 giây và tạo 64 keyword groups, cho thấy pipeline đang over-extract keyword và làm LLM2 xử lý quá nặng.

### Next Action
Test với nhiều tham số khác nhau để tối ưu. 

## Performance Comparison Table

In [4]:
performance_df = summary_df.copy()
if not performance_df.empty:
    performance_df["Seconds per page"] = (
        performance_df["Processing time (s)"] / performance_df["Pages"].replace(0, pd.NA)
    ).round(2)
    performance_df["Seconds per segment"] = (
        performance_df["Processing time (s)"] / performance_df["Segments"].replace(0, pd.NA)
    ).round(2)
    performance_df["Seconds per keyword group"] = (
        performance_df["Processing time (s)"] / performance_df["Keyword groups"].replace(0, pd.NA)
    ).round(2)
    performance_df = performance_df[
        [
            "Result file",
            "Document",
            "Processing time (s)",
            "Pages",
            "Segments",
            "Keyword groups",
            "Seconds per page",
            "Seconds per segment",
            "Seconds per keyword group",
        ]
    ].sort_values("Processing time (s)")

display(performance_df)

,Result file,Document,Processing time (s),Pages,Segments,Keyword groups,Seconds per page,Seconds per segment,Seconds per keyword group
3,4_result.json,4.pdf,44.63,2,12,12,22.32,3.72,3.72
2,2_result.json,2.pdf,49.62,2,15,11,24.81,3.31,4.51
6,agape_atp_supply_result.json,agape_atp_supply.txt,50.30,1,5,8,50.30,10.06,6.29
0,1_original_long_result.json,1_original_long.pdf,96.36,5,64,19,19.27,1.51,5.07
1,1_result.json,1.pdf,110.66,2,14,18,55.33,7.90,6.15
5,6_result.json,6.pdf,168.69,5,45,15,33.74,3.75,11.25
4,5_result.json,5.pdf,413.76,16,91,64,25.86,4.55,6.46
